# Part 2. Optical Image Check and Download

This notebook searches Sentinel-2 L2A images over Fox Glacier, filters scenes by AOI cloud/snow fraction, saves quicklook PNGs, and downloads the selected B08 GeoTIFFs. Run it from the lab folder containing `fox_glacier_from_osm.geojson`.


## Colab setup

Run this first in Google Colab. Local users can skip it if the required packages are already installed.


In [1]:
# Colab setup for Part 2: STAC search + raster reading/writing.
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
NEEDED = ["numpy", "rasterio", "matplotlib", "pystac_client", "planetary_computer"]
missing = [m for m in NEEDED if importlib.util.find_spec(m) is None]

if IN_COLAB and missing:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "numpy",
            "matplotlib",
            "rasterio",
            "pystac-client",
            "planetary-computer",
        ],
        check=True,
    )
else:
    print("Part 2 environment looks ready, or this is not Colab.")


Part 2 environment looks ready, or this is not Colab.


In [2]:
from __future__ import annotations

import json
import re
import sys
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Iterator

import numpy as np
import rasterio
from matplotlib import pyplot as plt
from rasterio.windows import from_bounds, transform as window_transform
from rasterio.warp import transform_bounds


## Settings

Edit the time window, cloud/snow thresholds, or download dates here if needed. Outputs are saved in the current folder.


In [3]:
# Current lab folder: all inputs and outputs stay here.
OPT_ROOT = Path.cwd().resolve()

FOX_GEOJSON = OPT_ROOT / "fox_glacier_from_osm.geojson"
STAC_PC = "https://planetarycomputer.microsoft.com/api/stac/v1"
QUICKLOOK_PNG = OPT_ROOT / "fox_glacier_b08_quicklook.png"
QUICKLOOK_PNG_DATES = OPT_ROOT / "fox_glacier_b08_quicklook_dates.png"

PRESET_GEOJSON = FOX_GEOJSON
PRESET_DATETIME = "2022-01-01T00:00:00Z/2026-12-31T23:59:59Z"
PRESET_BUFFER_DEG = 0.02
PRESET_SCL_CLOUD_CLASSES = (3, 8, 9, 10)
PRESET_SCL_SNOW_CLASSES = (11,)
PRESET_MAX_AOI_CLOUD_PCT = 10.0
PRESET_MAX_AOI_SNOW_PCT = 40.0
PRESET_STAC_SCENE_CLOUD_LT: float | None = None
PRESET_LIMIT: int | None = None
PRESET_DOWNLOAD_DIR = OPT_ROOT


In [4]:
def _load_dotenv() -> None:
    # If python-dotenv is installed, load .env (e.g. PC-related vars)
    try:
        from dotenv import load_dotenv
    except ImportError:
        return
    p = OPT_ROOT / ".env"
    if p.is_file():
        load_dotenv(p)


def _iter_lonlat_pairs(coords: Any) -> Iterator[tuple[float, float]]:
    # Walk GeoJSON geometry coordinates recursively; yield (lon, lat)
    if isinstance(coords, (int, float)):
        return
    if isinstance(coords, list):
        if len(coords) >= 2 and all(isinstance(x, (int, float)) for x in coords[:2]):
            yield float(coords[0]), float(coords[1])
            return
        for part in coords:
            yield from _iter_lonlat_pairs(part)


def bbox_from_geojson(path: Path, buffer_deg: float) -> tuple[float, float, float, float]:
    # WGS84 bbox (minx, miny, maxx, maxy): axis-aligned bounds + buffer
    data = json.loads(path.read_text(encoding="utf-8"))
    feats = data.get("features")
    if feats is None and data.get("type") == "Feature":
        feats = [data]
    elif feats is None:
        raise ValueError("GeoJSON must be a Feature or FeatureCollection")
    lons, lats = [], []
    for feat in feats:
        for lon, lat in _iter_lonlat_pairs((feat.get("geometry") or {}).get("coordinates")):
            lons.append(lon)
            lats.append(lat)
    if not lons:
        raise ValueError("No coordinates found in GeoJSON")
    b = buffer_deg
    return min(lons) - b, min(lats) - b, max(lons) + b, max(lats) + b


In [5]:
def parse_ymd(s: str) -> str:
    # Strict YYYY-MM-DD (same as CLI script)
    datetime.strptime(s.strip(), "%Y-%m-%d")
    return s.strip()


def _item_utc_date(item) -> date:
    # Observation calendar day in UTC from pystac Item (for per-day download matching)
    dt = item.datetime
    if isinstance(dt, str):
        dt = datetime.fromisoformat(dt.replace("Z", "+00:00"))
    if dt is not None:
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc).date()
    raw = item.properties.get("datetime")
    if not raw:
        raise ValueError("item has no datetime")
    dtp = datetime.fromisoformat(str(raw).replace("Z", "+00:00"))
    if dtp.tzinfo is None:
        dtp = dtp.replace(tzinfo=timezone.utc)
    return dtp.astimezone(timezone.utc).date()


In [6]:
def _rio_env():
    # Fewer GDAL dir scans; restrict VSICURL extensions for cloud reads
    return rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.TIF,.jp2,.JP2,.xml",
    )


def _read_band1_window_masked(source: str | Path, bbox4326: tuple[float, float, float, float]):
    # Band 1 window inside bbox; None if all masked/invalid
    minx, miny, maxx, maxy = bbox4326
    with _rio_env(), rasterio.open(source) as src:
        if src.crs is None:
            return None
        lb = transform_bounds("EPSG:4326", src.crs, minx, miny, maxx, maxy)
        win = from_bounds(*lb, transform=src.transform)
        arr = src.read(1, window=win, boundless=True, masked=True)
    if isinstance(arr, np.ma.MaskedArray) and not np.any(~arr.mask):
        return None
    return arr


def read_b08_chip_any(source: str | Path, bbox4326: tuple[float, float, float, float]):
    # B08 reflectance chip over Fox bbox as float32; None on failure
    arr = _read_band1_window_masked(source, bbox4326)
    return None if arr is None else arr.astype("float32")


def scl_aoi_cloud_snow_percents(item, bbox4326, cloud_classes: tuple[int, ...], snow_classes: tuple[int, ...]):
    # AOI SCL cloud/snow fractions; (None, None) if read fails
    if "SCL" not in item.assets:
        return None, None
    try:
        arr = _read_band1_window_masked(item.assets["SCL"].href, bbox4326)
    except Exception:
        return None, None
    if arr is None:
        return None, None
    if isinstance(arr, np.ma.MaskedArray):
        data = np.asarray(arr.data)
        valid = ~arr.mask & (data > 0)
    else:
        data = np.asarray(arr)
        valid = data > 0
    if not np.any(valid):
        return None, None
    d = float(np.sum(valid))
    c = 100.0 * np.sum(np.isin(data, np.asarray(cloud_classes, dtype=data.dtype)) & valid) / d
    s = (
        100.0 * np.sum(np.isin(data, np.asarray(snow_classes, dtype=data.dtype)) & valid) / d
        if snow_classes
        else 0.0
    )
    return float(c), float(s)


In [7]:
def _title_from_item_id(item_id: str) -> str:
    # Parse YYYY-MM-DD from S2 L2A item id for subplot titles
    m = re.search(r"_(\d{8})T", item_id)
    if not m or len(m.group(1)) < 8:
        return item_id[:16]
    raw = m.group(1)
    return f"{raw[:4]}-{raw[4:6]}-{raw[6:8]}"


def _chip_to_u8(chip, max_side_px: int = 512):
    # 2–98% linear stretch to uint8; downsample first when very large
    h, w = chip.shape
    step = max(1, max(h, w) // max_side_px)
    small = chip[::step, ::step]
    v = small.compressed() if isinstance(small, np.ma.MaskedArray) else small.ravel()
    v = v[np.isfinite(v)]
    if v.size == 0:
        return None
    lo, hi = np.percentile(v, (2.0, 98.0))
    hi = max(hi, lo + 1e-6)
    base = np.ma.filled(small, lo) if isinstance(small, np.ma.MaskedArray) else np.nan_to_num(small, nan=lo)
    return np.clip((base - lo) / (hi - lo) * 255.0, 0, 255).astype("uint8")


def write_fox_b08_quicklook_chips(chips: list[tuple[str, Any]], out_png: Path, *, suptitle: str) -> None:
    # Mosaic multiple grayscale panels into one PNG
    chips = [(t, u) for t, u in chips if u is not None]
    if not chips:
        print("  quicklook: no valid chips", file=sys.stderr)
        return
    n, ncols = len(chips), min(5, len(chips))
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows), squeeze=False)
    for i, (title, u8) in enumerate(chips):
        r, c = divmod(i, ncols)
        axes[r][c].imshow(u8, cmap="gray", interpolation="nearest")
        axes[r][c].set_title(title, fontsize=9)
        axes[r][c].axis("off")
    for ax in axes.flat[len(chips) :]:
        ax.axis("off")
    fig.suptitle(suptitle, fontsize=11)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"--- Quicklook saved: {out_png} ({n} panel(s)) ---")


def write_fox_b08_quicklook_from_paths(
    b08_paths: list[Path],
    bbox4326: tuple[float, float, float, float],
    out_png: Path,
    *,
    suptitle: str = "S2 B08 Fox crop (local)",
    stem_titles: bool = False,
) -> None:
    # Second quicklook from local GeoTIFFs (after date-specific downloads)
    chips = []
    for p in b08_paths:
        chip = read_b08_chip_any(p, bbox4326)
        if chip is None or chip.size == 0:
            print(f"  quicklook skip: {p.name}", file=sys.stderr)
            continue
        u8 = _chip_to_u8(chip)
        if u8 is not None:
            chips.append((p.stem if stem_titles else _title_from_item_id(p.name), u8))
    write_fox_b08_quicklook_chips(chips, out_png, suptitle=suptitle)


In [8]:
def write_b08_fox_chip_geotiff(item, bbox4326: tuple[float, float, float, float], dest: Path) -> Path | None:
    # Open B08 asset over HTTP(S), window-read, write single-band GeoTIFF
    if "B08" not in item.assets:
        return None
    url, minx, miny, maxx, maxy = item.assets["B08"].href, *bbox4326
    with _rio_env(), rasterio.open(url) as src:
        if src.crs is None:
            return None
        lb = transform_bounds("EPSG:4326", src.crs, minx, miny, maxx, maxy)
        win = from_bounds(*lb, transform=src.transform)
        arr = src.read(1, window=win, boundless=True, masked=True)
        transform = window_transform(win, src.transform)
        h, w = int(arr.shape[0]), int(arr.shape[1])
        if h < 1 or w < 1:
            return None
        dtype = arr.dtype if not isinstance(arr, np.ma.MaskedArray) else arr.data.dtype
        nodata = src.nodata
        profile: dict[str, Any] = {
            "driver": "GTiff",
            "height": h,
            "width": w,
            "count": 1,
            "dtype": dtype,
            "crs": src.crs,
            "transform": transform,
            "compress": "lzw",
        }
        if w >= 256 and h >= 256:
            profile.update(tiled=True, blockxsize=256, blockysize=256)
        if nodata is not None:
            profile["nodata"] = nodata
        data = np.ma.filled(arr, nodata if nodata is not None else 0) if isinstance(arr, np.ma.MaskedArray) else arr
        dest.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(dest, "w", **profile) as dst:
            dst.write(data, 1)
    return dest


In [9]:
def _filter_s2(items_raw, bbox, max_c, max_s):
    # Filter by PRESET_SCL_* and thresholds; returns (scored list, drop counts)
    scored, rc, rs, rf = [], 0, 0, 0
    for it in items_raw:
        c, s = scl_aoi_cloud_snow_percents(it, bbox, PRESET_SCL_CLOUD_CLASSES, PRESET_SCL_SNOW_CLASSES)
        if c is None or s is None:
            rf += 1
        elif c > max_c:
            rc += 1
        elif s > max_s:
            rs += 1
        else:
            scored.append((it, c, s, it.properties.get("eo:cloud_cover")))
    return scored, (rc, rs, rf)


def _quicklook_http(items, bbox, png: Path, *, suptitle: str) -> None:
    # For each AOI-pass item, HTTP-read B08 chip and build mosaic
    chips = []
    for i, item in enumerate(items):
        if "B08" not in item.assets:
            continue
        try:
            chip = read_b08_chip_any(item.assets["B08"].href, bbox)
        except Exception as e:
            print(f"  chip FAIL {i + 1} {item.id}: {e}", file=sys.stderr)
            continue
        u8 = _chip_to_u8(chip) if chip is not None else None
        if u8 is not None:
            chips.append((_title_from_item_id(item.id), u8))
    write_fox_b08_quicklook_chips(chips, png, suptitle=suptitle)


In [10]:
def run_s2(
    bbox: tuple[float, float, float, float],
    datetime_str: str,
    *,
    mode: str,
    pick_dates: list[str] | None,
    download_dir: Path,
) -> list[Path]:
    # mode=default: first quicklook only; else GeoTIFFs + second PNG
    try:
        from pystac_client import Client
        import planetary_computer
    except ImportError as e:
        raise SystemExit(f"Install requirements first: pip install -r requirements.txt\n{e}") from e

    downloaded: list[Path] = []
    cat = Client.open(STAC_PC, modifier=planetary_computer.sign_inplace)
    mc, ms = PRESET_MAX_AOI_CLOUD_PCT, PRESET_MAX_AOI_SNOW_PCT
    print(f"STAC {STAC_PC}\nBBox {bbox}\nTime {datetime_str}\nAOI SCL cloud≤{mc}% snow≤{ms}%\n")

    q: dict[str, Any] = {}
    if PRESET_STAC_SCENE_CLOUD_LT is not None:
        q["eo:cloud_cover"] = {"lt": PRESET_STAC_SCENE_CLOUD_LT}
    kw: dict[str, Any] = dict(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=datetime_str,
        sortby=[{"field": "datetime", "direction": "asc"}],
        max_items=PRESET_LIMIT,
    )
    if q:
        kw["query"] = q
    raw = list(cat.search(**kw).items())
    print(f"--- AOI filter {PRESET_SCL_CLOUD_CLASSES} / {PRESET_SCL_SNOW_CLASSES} ---", flush=True)
    scored, (rc, rs, rf) = _filter_s2(raw, bbox, mc, ms)
    print(f"  {len(raw)} -> {len(scored)} (drop cloud:{rc} snow:{rs} read:{rf})", flush=True)
    items = [t[0] for t in scored]
    cap = f" (max {PRESET_LIMIT})" if PRESET_LIMIT is not None else ""
    print(f"=== sentinel-2-l2a ({len(items)} granules{cap}) ===")
    for n, (it, ac, asn, scc) in enumerate(scored, 1):
        dt0 = it.properties.get("datetime", it.datetime)
        print(f"  {n:3d}  {dt0}  cloud={ac:.1f}% snow={asn:.1f}% scene_cc={scc}  {it.id}")
    if not items:
        print("  (no items — edit PRESET_DATETIME or PRESET_MAX_AOI_* )\n")
        return downloaded
    print()

    print("--- Quicklook 1/2: all AOI-clear (HTTP) ---")
    _quicklook_http(items, bbox, QUICKLOOK_PNG, suptitle="S2 B08 all AOI-clear (Fox bbox, HTTP)")

    if mode == "default":
        print(f"\n-> {QUICKLOOK_PNG}\nTip: set DOWNLOAD_DATES for Fox-crop + {QUICKLOOK_PNG_DATES.name}\n")
        return downloaded

    assert pick_dates is not None
    print("\n--- Fox-crop GeoTIFF ---")
    by_day: dict[date, list[Any]] = {}
    for it, *_ in scored:
        try:
            by_day.setdefault(_item_utc_date(it), []).append(it)
        except ValueError:
            pass
    for ds in pick_dates:
        d = date.fromisoformat(ds)
        matches = by_day.get(d, [])
        if not matches:
            print(f"  {d}: no granule")
            continue
        for j, it in enumerate(matches):
            name = f"B08_fox_{d.isoformat()}.tif" if len(matches) == 1 else f"B08_fox_{d.isoformat()}_{j + 1}.tif"
            try:
                dest = download_dir / name
                if dest.is_file():
                    print(f"  {dest.name} already exists")
                    downloaded.append(dest)
                    continue
                p = write_b08_fox_chip_geotiff(it, bbox, dest)
                if p is None:
                    print(f"  skip {d} no B08")
                else:
                    print(f"  {p.name} ({p.stat().st_size / (1024 * 1024):.2f} MiB)  {it.id}")
                    downloaded.append(p)
            except Exception as e:
                print(f"  FAIL {d} {it.id}: {e}", file=sys.stderr)
    print()
    if downloaded:
        print("--- Quicklook 2/2: selected dates (local GeoTIFF) ---")
        write_fox_b08_quicklook_from_paths(
            downloaded,
            bbox,
            QUICKLOOK_PNG_DATES,
            suptitle="S2 B08 selected dates (Fox crop)",
            stem_titles=True,
        )
        print(f"PNG: {QUICKLOOK_PNG} + {QUICKLOOK_PNG_DATES}\n")
    else:
        print(f"(no GeoTIFF for 2/2); see {QUICKLOOK_PNG}\n", file=sys.stderr)

    return downloaded


## Select dates

These two dates are used for the feature-tracking pair in Part 3.


In [11]:
# YYYYMMDD or YYYY-MM-DD strings
DOWNLOAD_DATES = ["20260301", "20260321"]


## Run search and download

This cell lists AOI-clear scenes, saves quicklook PNGs, and downloads the selected B08 GeoTIFFs.


In [12]:
def _norm_date(s: str) -> str:
    # Normalize to YYYY-MM-DD for date.fromisoformat (notebook convenience)
    t = str(s).strip().replace("-", "")
    if len(t) != 8 or not t.isdigit():
        raise ValueError(f"bad date: {s!r}")
    datetime.strptime(t, "%Y%m%d")
    return f"{t[:4]}-{t[4:6]}-{t[6:8]}"


_load_dotenv()
if not PRESET_GEOJSON.is_file():
    raise FileNotFoundError(PRESET_GEOJSON)

bbox = bbox_from_geojson(PRESET_GEOJSON, PRESET_BUFFER_DEG)
dates = [_norm_date(x) for x in DOWNLOAD_DATES] if DOWNLOAD_DATES else []

print(f"OPT_ROOT={OPT_ROOT}\nGeoJSON={PRESET_GEOJSON}\nOut={PRESET_DOWNLOAD_DIR}\nDates={dates or '(none)'}\n")

run_s2(
    bbox,
    PRESET_DATETIME,
    mode="download_dates" if dates else "default",
    pick_dates=dates if dates else None,
    download_dir=PRESET_DOWNLOAD_DIR,
)


OPT_ROOT=/home/xguo/Remote-Sense-Application/FertureTrackingLab
GeoJSON=/home/xguo/Remote-Sense-Application/FertureTrackingLab/fox_glacier_from_osm.geojson
Out=/home/xguo/Remote-Sense-Application/FertureTrackingLab
Dates=['2026-03-01', '2026-03-21']

STAC https://planetarycomputer.microsoft.com/api/stac/v1
BBox (170.04384589999998, -43.555430300000005, 170.18953140000002, -43.4789855)
Time 2022-01-01T00:00:00Z/2026-12-31T23:59:59Z
AOI SCL cloud≤10.0% snow≤40.0%

--- AOI filter (3, 8, 9, 10) / (11,) ---
  383 -> 40 (drop cloud:290 snow:53 read:0)
=== sentinel-2-l2a (40 granules) ===
    1  2022-02-20T22:37:11.024000Z  cloud=9.8% snow=38.9% scene_cc=5.525804  S2A_MSIL2A_20220220T223711_R072_T59GMM_20240517T165931
    2  2022-03-27T22:37:09.024000Z  cloud=0.0% snow=35.6% scene_cc=17.842379  S2B_MSIL2A_20220327T223709_R072_T59GMM_20220328T093821
    3  2022-04-01T22:37:11.024000Z  cloud=2.5% snow=34.9% scene_cc=4.326972  S2A_MSIL2A_20220401T223711_R072_T59GMM_20220402T083542
    4  2022-04

[PosixPath('/home/xguo/Remote-Sense-Application/FertureTrackingLab/B08_fox_2026-03-01.tif'),
 PosixPath('/home/xguo/Remote-Sense-Application/FertureTrackingLab/B08_fox_2026-03-21.tif')]